# 실험 결과 비교, 평균 ± 표준편차, paired t-test

In [1]:
import os

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

STEP = 10
GUIDANCE_SCALES = [2]  # 비교에 쓸 classifier guidance scale (예: [0.5, 1, 2, 4, 10]으로 늘리면 scale마다 열이 생긴다)
def cg_label(sc):
    return "exp2_classifier_guidance" if len(GUIDANCE_SCALES) == 1 else f"exp2_classifier_guidance_s{sc:g}"


SOURCES = {  # 라벨 → (CSV 경로, 이 CSV에서 쓸 method 정규식)
    "exp0_baseline": ("exp0_baseline/exp0_baseline.csv", r"^0_baseline"),
    "exp1_oversampling": ("exp1_oversampling/exp1_oversampling.csv",
                          rf"^1_diffusion_oversampling_ddpm1000_step{STEP}$"),
}
CG_CSV = "exp2_classifier_guidance/exp2_classifier_guidance.csv"
CLASS_NAMES = {"0": "Normal", "1": "Underfill_50FR", "2": "Underfill_Fan", "macro": "macro"}

frames = []
for label, (path, pat) in SOURCES.items():
    if not os.path.exists(path):
        print("결과 CSV 없음, 건너뜀:", path); continue
    d = pd.read_csv(path)
    d = d[d["method"].str.contains(pat, regex=True)].assign(method=label)
    frames.append(d)
if os.path.exists(CG_CSV):
    d = pd.read_csv(CG_CSV)
    m = d["method"].str.extract(rf"step{STEP}_s([\d.]+)$")[0]
    for sc in GUIDANCE_SCALES:
        frames.append(d[m == f"{sc:g}"].assign(method=cg_label(sc)))
else:
    print("결과 CSV 없음, 건너뜀:", CG_CSV)

df_all = pd.concat(frames, ignore_index=True)
df_all["class"] = df_all["class"].astype(str)
df_all = df_all.drop_duplicates(subset=["method", "seed", "class"], keep="last")  # 이어서 돌려 중복된 행은 마지막 것만
METHOD_ORDER = [m for m in ["exp0_baseline", "exp1_oversampling"] if m in set(df_all["method"])]
METHOD_ORDER += [cg_label(sc) for sc in GUIDANCE_SCALES if cg_label(sc) in set(df_all["method"])]
print("실험별 seed 수:")
print(df_all[df_all["class"] == "macro"].groupby("method")["seed"].nunique().reindex(METHOD_ORDER).to_string())

실험별 seed 수:
method
exp0_baseline               30
exp1_oversampling           30
exp2_classifier_guidance    30


## 1. 실험별 평균 ± 표준편차

In [2]:
def per_seed(metric, cls="macro"):
    sub = df_all[df_all["class"] == str(cls)]
    return sub.pivot_table(index="seed", columns="method", values=metric, aggfunc="mean")[METHOD_ORDER]


summary_items = [("accuracy", "macro"), ("precision", "macro"), ("recall", "macro"), ("f1", "macro")]
summary = {}
for metric, cls in summary_items:
    tab = per_seed(metric, cls)
    name = f"{metric}" if cls == "macro" else f"{metric}_{CLASS_NAMES[cls]}"
    summary[name] = tab.mean().map("{:.3f}".format) + " ± " + tab.std(ddof=1).map("{:.3f}".format)
summary = pd.DataFrame(summary)
summary.insert(0, "n_seeds", per_seed("f1").notna().sum())
summary

,n_seeds,accuracy,precision,recall,f1
method,,,,,
exp0_baseline,30,0.751 ± 0.038,0.817 ± 0.024,0.751 ± 0.038,0.722 ± 0.052
exp1_oversampling,30,0.795 ± 0.052,0.831 ± 0.031,0.795 ± 0.052,0.782 ± 0.066
exp2_classifier_guidance,30,0.812 ± 0.052,0.840 ± 0.033,0.812 ± 0.052,0.804 ± 0.063


## 2. paired t-test

In [3]:
CG = [m for m in METHOD_ORDER if m.startswith("exp2_classifier_guidance")]
COMPARISONS = [("exp1_oversampling", "exp0_baseline", "오버샘플링 vs baseline")]
COMPARISONS += [(m, "exp1_oversampling", "classifier guidance vs 오버샘플링") for m in CG]
COMPARISONS += [(m, "exp0_baseline", "classifier guidance vs baseline") for m in CG]
COMPARISONS = [c for c in COMPARISONS if c[0] in METHOD_ORDER and c[1] in METHOD_ORDER]


def paired_tests(metric="f1", cls="macro", comparisons=COMPARISONS, df=None):
    df = df_all if df is None else df
    sub = df[df["class"] == str(cls)]
    table = sub.pivot_table(index="seed", columns="method", values=metric, aggfunc="mean")
    rows = []
    for a, b, desc in comparisons:
        if a not in table or b not in table:
            continue
        pair = table[[a, b]].dropna()
        n = len(pair)
        diff = pair[a] - pair[b]
        row = {"비교": desc, "A": a, "B": b, "n": n,
               "mean_A": pair[a].mean(), "mean_B": pair[b].mean()}
        if n >= 2 and diff.std(ddof=1) > 0:
            t_stat, p_val = stats.ttest_rel(pair[a], pair[b])
            row.update({"t": t_stat, "p": p_val})
        else:
            row.update({"t": np.nan, "p": np.nan})
        rows.append(row)
    res = pd.DataFrame(rows)
    if res.empty:
        print(f"[{metric}/{cls}] 비교할 수 있는 실험 쌍이 아직 없습니다.")
        return res
    res["유의(p<0.05)"] = res["p"] < 0.05
    return res


pd.set_option("display.float_format", "{:.4f}".format)
paired_tests("f1", "macro")

,비교,A,B,n,mean_A,mean_B,t,p,유의(p<0.05)
0,오버샘플링 vs baseline,exp1_oversampling,exp0_baseline,30,0.7821,0.7219,3.9631,0.0004,True
1,classifier guidance vs 오버샘플링,exp2_classifier_guidance,exp1_oversampling,30,0.8036,0.7821,2.1621,0.0390,True
2,classifier guidance vs baseline,exp2_classifier_guidance,exp0_baseline,30,0.8036,0.7219,5.3406,0.0000,True
